In [0]:
# ===================================================
# BLOCK 1 — PARAMETERS AND TABLES (PYTHON)
# ===================================================

"""
Define the execution context and governed datasets used by the operational
quality gate.
"""

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("job_run_id", "MANUAL", "Lakeflow Job Run ID")
dbutils.widgets.text(
    "minimum_bronze_rows",
    "1",
    "Minimum acceptable Bronze rows",
)

JOB_RUN_ID = dbutils.widgets.get("job_run_id")
MINIMUM_BRONZE_ROWS = int(
    dbutils.widgets.get("minimum_bronze_rows")
)

ARRIVAL_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/"
    "external_source/streaming_demo/input"
)

BRONZE_TABLE = (
    "semiconplus_portfolio.bronze.streaming_test_results"
)
SILVER_TABLE = (
    "semiconplus_portfolio.silver.streaming_test_results"
)
LATE_TABLE = (
    "semiconplus_portfolio.silver.streaming_late_test_results"
)
QUARANTINE_TABLE = (
    "semiconplus_portfolio.quarantine.streaming_test_results"
)
GOLD_TABLE = (
    "semiconplus_portfolio.gold.mart_streaming_yield_5m"
)

In [0]:
# ===================================================
# BLOCK 2 — LOAD DATASETS AND COUNTS (PYTHON)
# ===================================================

"""
Materialize exact operational counts once and reuse them across reconciliation,
audit logging, and downstream task values.
"""

source_df = (
    spark.read
    .option("multiLine", "false")
    .json(ARRIVAL_DIRECTORY)
)

bronze_df = spark.table(BRONZE_TABLE)
silver_df = spark.table(SILVER_TABLE)
late_df = spark.table(LATE_TABLE)
quarantine_df = spark.table(QUARANTINE_TABLE)
gold_df = spark.table(GOLD_TABLE)

source_count = source_df.count()
bronze_count = bronze_df.count()
silver_count = silver_df.count()
late_count = late_df.count()
quarantine_count = quarantine_df.count()
gold_count = gold_df.count()

count_results = {
    "source_rows": source_count,
    "bronze_rows": bronze_count,
    "silver_rows": silver_count,
    "late_rows": late_count,
    "quarantine_rows": quarantine_count,
    "gold_rows": gold_count,
}

display(
    spark.createDataFrame(
        [
            (metric_name, metric_value)
            for metric_name, metric_value in count_results.items()
        ],
        ["metric_name", "metric_value"],
    )
)

In [0]:
# ===================================================
# BLOCK 3 — SOURCE-TO-BRONZE RECONCILIATION (PYTHON)
# ===================================================

"""
Confirm that all staged files were ingested exactly once and that Bronze
contains the complete immutable source population.
"""

source_file_count = len(
    [
        item
        for item in dbutils.fs.ls(ARRIVAL_DIRECTORY)
        if not item.isDir() and item.name.lower().endswith(".json")
    ]
)

bronze_file_count = (
    bronze_df
    .select("_source_file_name")
    .distinct()
    .count()
)

assert bronze_count >= MINIMUM_BRONZE_ROWS, (
    f"Bronze contains {bronze_count} rows; "
    f"minimum required is {MINIMUM_BRONZE_ROWS}."
)

assert source_count == bronze_count, (
    f"Source/Bronze mismatch: source={source_count}, "
    f"bronze={bronze_count}."
)

assert source_file_count == bronze_file_count, (
    f"Source/Bronze file mismatch: source={source_file_count}, "
    f"bronze={bronze_file_count}."
)

In [0]:
# ===================================================
# BLOCK 4 — SILVER UNIQUENESS AND ROUTING (PYTHON)
# ===================================================

"""
Validate that accepted event identifiers remain unique and that all published
Silver records originate from the governed Bronze population.
"""

duplicate_silver_ids = (
    silver_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_silver_ids == 0, (
    f"Accepted Silver contains {duplicate_silver_ids} duplicate event IDs."
)

assert silver_count <= bronze_count
assert late_count <= bronze_count
assert quarantine_count <= bronze_count
assert silver_count > 0

In [0]:
# ===================================================
# BLOCK 5 — LATE-DATA VALIDATION (PYTHON)
# ===================================================

"""
Confirm that deliberately delayed events were routed to the late-data table and
did not enter the accepted low-latency reporting population.
"""

unflagged_late_rows = late_df.filter(
    F.col("is_deliberately_late") != F.lit(True)
).count()

late_ids_in_accepted_silver = (
    late_df.select("event_id")
    .join(
        silver_df.select("event_id"),
        on="event_id",
        how="inner",
    )
    .count()
)

assert late_count > 0, "Expected controlled late records."
assert unflagged_late_rows == 0
assert late_ids_in_accepted_silver == 0

In [0]:
# ===================================================
# BLOCK 6 — QUARANTINE EXPLAINABILITY (PYTHON)
# ===================================================

"""
Require every quarantined record to contain at least one actionable failure
reason. An empty quarantine table is accepted.
"""

unexplained_quarantine_rows = quarantine_df.filter(
    F.col("_quality_reasons").isNull()
    | (F.size("_quality_reasons") == 0)
).count()

assert unexplained_quarantine_rows == 0

In [0]:
# ===================================================
# BLOCK 7 — GOLD-MART VALIDATION (PYTHON)
# ===================================================

"""
Validate the five-minute Gold mart at its complete business grain and enforce
required keys, nonnegative measures, status reconciliation, and valid yield
ranges before the dataset is exposed to reporting consumers.
"""

GOLD_KEY_COLUMNS = [
    "window_start_utc",
    "site_id",
    "equipment_id",
    "product_group_id",
    "device_id",
]

duplicate_gold_keys = (
    gold_df
    .groupBy(*GOLD_KEY_COLUMNS)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_gold_rows = gold_df.filter(
    F.col("window_start_utc").isNull()
    | F.col("window_end_utc").isNull()
    | F.col("site_id").isNull()
    | F.col("equipment_id").isNull()
    | F.col("product_group_id").isNull()
    | F.col("device_id").isNull()
    | (F.col("window_end_utc") <= F.col("window_start_utc"))
    | (F.col("event_count") <= 0)
    | (F.col("pass_count") < 0)
    | (F.col("fail_count") < 0)
    | (F.col("alarm_count") < 0)
    | (F.col("pass_count") > F.col("event_count"))
    | (F.col("fail_count") > F.col("event_count"))
    | (F.col("alarm_count") > F.col("event_count"))
    | (
        F.col("pass_count")
        + F.col("fail_count")
        + F.col("alarm_count")
        != F.col("event_count")
    )
    | (
        F.col("yield_rate").isNotNull()
        & (
            (F.col("yield_rate") < 0)
            | (F.col("yield_rate") > 1)
        )
    )
    | (
        F.col("average_test_time_seconds").isNotNull()
        & (F.col("average_test_time_seconds") < 0)
    )
    | (
        F.col("maximum_test_time_seconds").isNotNull()
        & (F.col("maximum_test_time_seconds") < 0)
    )
    | (
        F.col("average_test_time_seconds").isNotNull()
        & F.col("maximum_test_time_seconds").isNotNull()
        & (
            F.col("average_test_time_seconds")
            > F.col("maximum_test_time_seconds")
        )
    )
).count()

display(
    spark.createDataFrame(
        [
            (
                gold_count,
                duplicate_gold_keys,
                invalid_gold_rows,
            )
        ],
        [
            "gold_rows",
            "duplicate_gold_keys",
            "invalid_gold_rows",
        ],
    )
)

assert gold_count > 0, "Gold mart contains no records."

assert duplicate_gold_keys == 0, (
    f"Gold mart contains {duplicate_gold_keys} duplicate business keys."
)

assert invalid_gold_rows == 0, (
    f"Gold mart contains {invalid_gold_rows} invalid metric records."
)

print("Gold mart validation passed.")

In [0]:
# ===================================================
# GOLD MART SCHEMA DIAGNOSTIC (PYTHON)
# ===================================================

gold_df = spark.table(
    "semiconplus_portfolio.gold.mart_streaming_yield_5m"
)

gold_df.printSchema()
display(gold_df.limit(5))

In [0]:
# ===================================================
# BLOCK 8 — PERSIST QUALITY METRICS (PYTHON)
# ===================================================

"""
Append the run's validation evidence to the operational metric table for trend
analysis, incident investigation, and portfolio documentation.
"""

recorded_at_utc = datetime.now(timezone.utc)

metric_records = [
    (
        JOB_RUN_ID,
        "source_rows",
        float(source_count),
        "source_rows = bronze_rows",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "bronze_rows",
        float(bronze_count),
        f"bronze_rows >= {MINIMUM_BRONZE_ROWS}",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "bronze_source_files",
        float(bronze_file_count),
        "bronze_source_files = 10",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "duplicate_silver_event_ids",
        float(duplicate_silver_ids),
        "duplicate_silver_event_ids = 0",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "late_rows",
        float(late_count),
        "late_rows > 0",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "unexplained_quarantine_rows",
        float(unexplained_quarantine_rows),
        "unexplained_quarantine_rows = 0",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "duplicate_gold_keys",
        float(duplicate_gold_keys),
        "duplicate_gold_keys = 0",
        "PASSED",
        recorded_at_utc,
    ),
    (
        JOB_RUN_ID,
        "invalid_gold_rows",
        float(invalid_gold_rows),
        "invalid_gold_rows = 0",
        "PASSED",
        recorded_at_utc,
    ),
]

metric_schema = T.StructType(
    [
        T.StructField("job_run_id", T.StringType(), False),
        T.StructField("metric_name", T.StringType(), False),
        T.StructField("metric_value", T.DoubleType(), True),
        T.StructField("expected_condition", T.StringType(), True),
        T.StructField("validation_status", T.StringType(), False),
        T.StructField("recorded_at_utc", T.TimestampType(), False),
    ]
)

metric_df = spark.createDataFrame(
    metric_records,
    metric_schema,
)

# Replace metrics for the same repaired run before appending refreshed results.
spark.sql(
    f"""
    DELETE FROM semiconplus_portfolio.operations.workflow_quality_metrics
    WHERE job_run_id = '{JOB_RUN_ID}'
    """
)

(
    metric_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "semiconplus_portfolio.operations.workflow_quality_metrics"
    )
)

In [0]:
# ===================================================
# BLOCK 9 — PUBLISH DOWNSTREAM TASK VALUES (PYTHON)
# ===================================================

"""
Publish validated row counts for the finalization task and the Lakeflow Job run
details.
"""

try:
    dbutils.jobs.taskValues.set(
        key="bronze_rows",
        value=bronze_count,
    )
    dbutils.jobs.taskValues.set(
        key="silver_rows",
        value=silver_count,
    )
    dbutils.jobs.taskValues.set(
        key="late_rows",
        value=late_count,
    )
    dbutils.jobs.taskValues.set(
        key="quarantine_rows",
        value=quarantine_count,
    )
    dbutils.jobs.taskValues.set(
        key="gold_rows",
        value=gold_count,
    )
except Exception:
    print("Task values are unavailable during interactive execution.")

print("QUALITY GATE: PASSED")